# Agentic Spine AI Pipeline — Colab Training

This notebook **trains the vision engine** (the imaging model the agentic
pipeline's `VisionEngine` agent uses) on a free Colab GPU, saves the
checkpoints to your Google Drive for later use, downloads them to your PC,
and optionally fits **calibrated probabilities**.

**What it trains (multi-task):**
- Landmark localization — L1–L5 vertebrae + L1/L2…L5/S1 discs, per-point confidence
- Disc degeneration (DDD) grade 0–4 — only when `ddd_labels.csv` is present
- Spondylolisthesis slip % — only when `spondy_labels.csv` is present

The full **agentic** layer (symptom agent, fusion agent, verification agent,
report agent) already works and calls Gemini — this notebook just gives the
pipeline its *eyes*.

**Before you start:** put your dataset in Google Drive — a folder
(default `MyDrive/dataset`) or a `dataset.zip`. Labels for DDD/spondylolisthesis
are optional (see cell 7).

## 0. Edit these to your project

Put your GitHub repo URL below (or leave it and clone locally another way).

In [ ]:
GITHUB_URL = 'https://github.com/codermisba/spinelit-ai.git'  # <-- EDIT: your repo
GEMINI_KEY = ''   # <-- OPTIONAL: set to enable LLM agents inside Colab tests
print('Repo:', GITHUB_URL)

In [ ]:
# 1. Check GPU (must show a GPU)
!nvidia-smi -L || echo 'Runtime > Change runtime type > GPU (T4 free)'

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone the repository
%cd /content
import os
GITHUB_URL = GITHUB_URL or 'https://github.com/codermisba/spinelit-ai.git'
if not os.path.exists('/content/spine-foundation'):
    !git clone {GITHUB_URL} spine-foundation
%cd /content/spine-foundation/
!git pull || echo '(already latest)'
print('cloned to /content/spine-foundation')

In [ ]:
# 4. Install dependencies (PyTorch + CUDA GPU is preinstalled on Colab)
!pip install -q -r requirements.txt
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

In [ ]:
# 5. Load the dataset from Google Drive (folder OR zip)
import os, shutil

DRIVE_ROOT = '/content/drive/MyDrive'
REPO_DATASET = '/content/spine-foundation/dataset'

# >>> EDIT if your folder lives somewhere else in Drive <<<
DATASET_DIR = f'{DRIVE_ROOT}/dataset'
DATASET_ZIP = f'{DRIVE_ROOT}/spine-foundation/dataset.zip'

def merge_copy(src, dst):
    count = 0
    for root, _, files in os.walk(src):
        target = os.path.normpath(os.path.join(dst, os.path.relpath(root, src)))
        os.makedirs(target, exist_ok=True)
        for name in files:
            shutil.copy2(os.path.join(root, name), os.path.join(target, name))
            count += 1
    return count

if os.path.isdir(DATASET_DIR):
    print('Copying dataset folder from Drive...')
    print(f'  {merge_copy(DATASET_DIR, REPO_DATASET)} files copied.')
elif os.path.exists(DATASET_ZIP):
    !unzip -q -o "{DATASET_ZIP}" -d "{REPO_DATASET}"
    print('Unzipped dataset.zip')
else:
    print('Dataset not found at:', DATASET_DIR)
    !ls "{DRIVE_ROOT}"
    raise FileNotFoundError('Edit DATASET_DIR in this cell to point at your Drive folder.')

print('\nTop level of dataset/:')
!ls "{REPO_DATASET}"

In [ ]:
# 6. Sanity check the dataset / model architecture
!python dataset.py && echo --- && python model.py

## 7. (Optional) Add grading labels for DDD + spondylolisthesis

If you have them, place these CSVs in Drive and they will be auto-detected.
Without labels, the pipeline trains localization + confidence only, and the
DDD/spondy heads fall back to the documented deterministic calibration.

- `dataset/ddd_labels.csv` → columns `filename,level,grade` (grade 0–4)
- `dataset/spondy_labels.csv` → columns `filename,level,slip_percent`
```
filename,level,grade
case_0000.jpg,L3/L4,3
...
```

In [ ]:
# 8. GPU-friendly settings (IMAGE_SIZE 512, bigger batch, more workers)
import re
cfg = open('config.py').read()
cfg = cfg.replace('IMAGE_SIZE = 256', 'IMAGE_SIZE = 512')
cfg = cfg.replace('BATCH_SIZE = 4', 'BATCH_SIZE = 32')
cfg = cfg.replace('NUM_WORKERS = 0', 'NUM_WORKERS = 2')
open('config.py','w').write(cfg)
print('Config updated for GPU training.')

# Optionally enable LLM agents inside this Colab (for live testing)
if GEMINI_KEY:
    import os
    os.environ['GEMINI_API_KEY'] = GEMINI_KEY
    open('/content/spine-foundation/.env','w').write('GEMINI_API_KEY=' + GEMINI_KEY + '\n')
    print('GEMINI_API_KEY set for Colab session.')

In [ ]:
# 9. TRAIN the vision engine (checkpoints land in checkpoints/)
!python train.py --epochs 60

In [ ]:
# 10. Evaluate on the validation split
!python evaluate.py

## 11. Optional: fit calibrated probabilities

If you provided DDD/spondy labels, this fits an isotonic calibrator that maps
raw model confidence to a **true probability (0–1)** — the accuracy guarantee
the agentic pipeline advertises. Saves `checkpoints/calibration.pkl`.

In [ ]:
import os
if os.path.exists('dataset/ddd_labels.csv') or os.path.exists('dataset/spondy_labels.csv'):
    !python train_calibrator.py
else:
    print('No grading labels present -> using deterministic fallback calibration. '
          'Re-run after adding ddd_labels.csv / spondy_labels.csv.')

## 12. Test the trained model through the agentic CLI

This runs the **real agentic pipeline** (vision + LLM agents). The reported
`agreement` / `calibrated probability` / verification reflect your trained model.
Set `GEMINI_KEY` in cell 0 to enable the LLM report; otherwise it degrades to the
numeric findings tables.

In [ ]:
import subprocess, glob
img = glob.glob('dataset/data/processed_tseg_jpgs/*.jpg')
img = img[0] if img else glob.glob('dataset/data/**/*.jpg', recursive=True)[0]
print('testing image:', img)
!python cli_pipeline.py --image "{img}" --age 58 --sex female \
    --pain-scale 6 --modality mri --pain-years 4 --start-year 2022 \
    --symptoms "low back pain and numbness down the right leg" --pretty

## 13. Optional: launch the Gradio agentic UI inside Colab
First cell returns a public `*.gradio.live` link — keep it open to demo.

In [ ]:
!python app.py --share

## 14. SAVE everything to Google Drive (survives runtime reset)
Copies the trained model + calibrator into Drive so you never lose them.

In [ ]:
import shutil, os
DEST = '/content/drive/MyDrive/spine-checkpoints'
os.makedirs(DEST, exist_ok=True)
saved = []
for f in ['best_model.pth', 'last_model.pth', 'calibration.pkl', 'longitudinal_model.pth']:
    src = f'checkpoints/{f}'
    if os.path.exists(src):
        shutil.copy2(src, DEST)
        saved.append(f)
        print('saved', src, '->', DEST)
if not saved:
    print('No checkpoints found yet - run the TRAIN cell first.')
else:
    print('\nSafe in Drive:', DEST)
    print('To restore later anywhere: copy these files back into checkpoints/')

## 15. Download the trained model to your PC

Run this, choose the files in the popup, and save them into
`E:\spine-foundation\checkpoints\` on your Windows machine. Then your local
`app.py` / `cli_pipeline.py` will show real numbers.

In [ ]:
from google.colab import files
import os
to_download = [f for f in ['best_model.pth','calibration.pkl']
               if os.path.exists(f'checkpoints/{f}')]
if to_download:
    files.download('checkpoints/' + to_download[0])
    if len(to_download) > 1:
        files.download('checkpoints/' + to_download[1])
else:
    print('Nothing to download yet - train first.')

## 16. Restore checkpoints from Drive anytime

If your Colab runtime reset or you moved to a new notebook, run this to bring
your trained models back into `checkpoints/`.

```python
import shutil, os
SRC = '/content/drive/MyDrive/spine-checkpoints'
os.makedirs('/content/spine-foundation/checkpoints', exist_ok=True)
for f in ['best_model.pth','last_model.pth','calibration.pkl','longitudinal_model.pth']:
    s = os.path.join(SRC, f)
    if os.path.exists(s):
        shutil.copy2(s, '/content/spine-foundation/checkpoints/' + f)
        print('restored', f)
print('done')
```